# Thinking Spatially: Mapping Health Facilities  (Solution)

Reference solution for `exercise_5b_health_facilities_mapping.ipynb` — **Lesson 5: GeoPandas**.

**Scenario.** You advise South Africa's health department. You have an OpenStreetMap export of
health facilities (hospitals, clinics, pharmacies, ...) and the official province boundaries. The
question: *how are facilities distributed across the nine provinces, and where are the gaps?*

**Data** (all local, under `data/0_raw/south_africa/geo_data/`)
- `south-africa.geojson` — OSM health facilities (points and building polygons)
- `zaf_admin_boundaries.geojson/zaf_admin1.geojson` — province boundaries (COD-AB)

**Skills:** table → GeoDataFrame · fixing mixed geometry · spatial join · choropleth vs density ·
multi-layer maps.

In [ ]:
pip install geopandas contextily mapclassify

In [ ]:
import sys
from pathlib import Path
import pandas as pd


import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))
from src.utilities.project_paths import RAW_DIR

GEO_DIR           = RAW_DIR / 'geo_data'
HEALTH_GEOJSON    = GEO_DIR / 'south-africa.geojson'
PROVINCES_GEOJSON = GEO_DIR / 'zaf_admin_boundaries.geojson' / 'zaf_admin1.geojson'

pd.set_option('display.float_format', '{:,.1f}'.format)
print('geopandas', gpd.__version__, '| contextily', cx.__version__)

---
# Part A — Loading Spatial Data

Two layers: the **facilities** (points) and the **provinces** (polygons). Along the way we hit a
real-world snag — the facilities layer mixes points and polygons — and fix it.

## A1. Load the health facilities

`gpd.read_file()` reads a GeoJSON straight into a `GeoDataFrame`. Inspect the CRS, the columns, and
the **geometry types**.

In [ ]:
health = gpd.read_file(HEALTH_GEOJSON)

print('CRS        :', health.crs)
print('rows, cols :', health.shape)
print('geom types :', health.geometry.geom_type.value_counts().to_dict())
print('top amenity:')
print(health['amenity'].value_counts().head(6))
display(health[['name', 'amenity', 'healthcare', 'operator']].head())

**Interpretation:** 3,055 facilities in `EPSG:4326`. Note the geometry is **mixed** — about 1,300
records are `Polygon` (the building footprint was mapped) and the rest are `Point`. A spatial join
needs points, so we will collapse everything to points in A3.

## A2. Load the province boundaries

The COD-AB file has one row per province. Keep the name (`adm1_name`), the area (`area_sqkm`), and
the geometry.

In [ ]:
provinces = gpd.read_file(PROVINCES_GEOJSON)[['adm1_name', 'adm1_pcode', 'area_sqkm', 'geometry']]

print('CRS   :', provinces.crs)
print('shape :', provinces.shape)
display(provinces[['adm1_name', 'area_sqkm']])

provinces.plot(figsize=(7, 7), color='lightgray', edgecolor='white')
plt.title('South African provinces')
plt.show()

## A3. Fix the mixed geometry, and tidy the facility type

Two clean-ups:
1. Collapse every facility to a single point with `.representative_point()` (a point guaranteed to
   lie inside the shape — works for both points and polygons).
2. Fold the messy `amenity` values into a small `facility_type` category.

In [ ]:
# 1. one point per facility (works whether the geometry is a Point or a Polygon)
health_points = health.copy()
health_points['geometry'] = health.geometry.representative_point()

# 2. tidy the amenity into a handful of categories
main_types = ['hospital', 'clinic', 'pharmacy', 'doctors', 'dentist']
health_points['facility_type'] = health_points['amenity'].where(
    health_points['amenity'].isin(main_types), 'other')

print('geom types now:', health_points.geometry.geom_type.value_counts().to_dict())
display(health_points['facility_type'].value_counts().to_frame())

ax = provinces.plot(figsize=(8, 8), color='lightgray', edgecolor='white')
health_points.plot(ax=ax, markersize=3, color='crimson')
plt.title('Health facilities over provinces')
plt.show()

**Interpretation:** After `.representative_point()` every row is a `Point`, so the layer is ready
for a spatial join. Hospitals, pharmacies and clinics dominate; a small `other` bucket collects
dentists' offices, labs and un-tagged rows.

---
# Part B — Spatial Join: Which Province Is Each Facility In?

The facilities have coordinates but no province label. A spatial join recovers it — *is this point
`within` that polygon?* — the "Spatial VLOOKUP".

## B1. Tag each facility with its province

In [ ]:
tagged = gpd.sjoin(health_points, provinces, how='left', predicate='within')

print('facilities :', len(health_points))
print('tagged rows:', len(tagged))
print('unmatched  :', tagged['adm1_name'].isna().sum())
display(tagged[['name', 'facility_type', 'adm1_name']].head())

**Interpretation:** Every facility fell inside exactly one province, so `adm1_name` was recovered
purely from location — no fragile text matching. `how='left'` would have exposed any facility that
landed in the ocean (a bad coordinate) as a `NaN`; here there are none.

## B2. Count facilities per province and per type

Now it is a plain DataFrame — use `groupby`.

In [ ]:
by_province = (tagged.groupby('adm1_name').size()
               .rename('n_facilities').reset_index()
               .sort_values('n_facilities', ascending=False))
display(by_province)

by_type = (tagged.groupby(['adm1_name', 'facility_type']).size()
           .unstack(fill_value=0))
display(by_type)

---
# Part C — Counts vs Density

Raw counts favour big, busy provinces. Normalising by area tells a different — and often fairer —
story.

## C1. Attach counts to the polygons and compute density

Merge the per-province counts back onto the province polygons, then compute facilities per
10,000 km² using `area_sqkm`.

In [ ]:
prov_map = provinces.merge(by_province, on='adm1_name', how='left')
prov_map['per_10000km2'] = prov_map['n_facilities'] / prov_map['area_sqkm'] * 10_000

display(
    prov_map[['adm1_name', 'n_facilities', 'area_sqkm', 'per_10000km2']]
    .sort_values('per_10000km2', ascending=False)
)

**Interpretation:** Gauteng leads on *both* count (1,145) and density (~630 per 10,000 km²) — it is
small and urban. But the ranking shifts otherwise: Western Cape has the 2nd-most facilities yet a
modest density because it is large, while the vast Northern Cape sits far below everyone on density
(~2 per 10,000 km²). Which map is "right" depends on the question you are answering.

---
# Part D — Multi-Layer Maps

Layer cake: basemap at the bottom, province choropleth in the middle, facility points on top.
Convert everything to **Web Mercator (EPSG:3857)** first so `contextily` basemaps align.

## D1. Facilities map — choropleth + points by type

In [ ]:
prov_web = prov_map.to_crs(3857)
pts_web  = health_points.to_crs(3857)

fig, ax = plt.subplots(figsize=(11, 11))

# Middle layer: choropleth of facility counts
prov_web.plot(ax=ax, column='n_facilities', cmap='YlGnBu', edgecolor='white', linewidth=0.6,
              legend=True, legend_kwds={'label': 'Facilities per province', 'shrink': 0.5},
              zorder=1)

# Top layer: facility points coloured by type
pts_web.plot(ax=ax, column='facility_type', markersize=6, cmap='Set1',
             legend=True, legend_kwds={'title': 'Facility type', 'loc': 'lower left', 'fontsize': 8},
             zorder=3)

# Bottom layer: basemap
try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
except Exception as e:
    print('Basemap skipped (offline?):', type(e).__name__)

ax.set_title('Health facilities across South Africa', fontsize=15, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## D2. Density map — the same data, normalised

Colour provinces by facilities per 10,000 km² instead of raw counts, and label each province.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

prov_web.plot(ax=ax, column='per_10000km2', cmap='OrRd', edgecolor='white', linewidth=0.6,
              legend=True, legend_kwds={'label': 'Facilities per 10,000 km2', 'shrink': 0.6},
              zorder=1)

for _, r in prov_web.iterrows():
    c = r.geometry.representative_point()
    ax.annotate(f"{r['adm1_name']}\n{r['per_10000km2']:.0f}", xy=(c.x, c.y),
                ha='center', fontsize=7, fontweight='bold')

try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
except Exception as e:
    print('Basemap skipped (offline?):', type(e).__name__)

ax.set_title('Facility density (per 10,000 km2)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

**Interpretation:** The count map shouts "Gauteng"; the density map keeps Gauteng on top but
reframes the rest — the interior provinces look far more thinly served once you account for their
size. Same numbers, different message. Always ask whether a choropleth should show totals or a
rate.